# Spark SQL · Mi cuaderno de consulta
**Jennifer Sánchez Richart · Fabric · 13 de noviembre de 2026**

Guía basada en tu notebook original y tus cuatro archivos CSV. Ejecuta de arriba abajo: primero configuración y carga, después las consultas que necesites. Los ejemplos usan los nombres y columnas reales de los archivos.

## Índice
1. Preparación y archivos
2. Explorar y seleccionar
3. Filtrar, transformar y tratar nulos
4. Agregar y unir tablas
5. CTE y funciones de ventana
6. Calidad de datos
7. Delta: escritura, historial y clonación (opcional)
8. Comparación con el endpoint SQL y solución de errores
9. Ejercicios y chuleta

## 1. Preparación y archivos
Estos son **CSV**, no libros Excel: son texto separado por comas, con cabecera y decimales con punto.

| Archivo / vista | Una fila representa | Clave | Filas originales |
|---|---|---|---:|
| clientes.csv / clientes | Un cliente | id_cliente | 500 |
| productos.csv / productos | Un producto | id_producto | 100 |
| pedidos.csv / pedidos | Un pedido | id_pedido | 5.000 |
| detalle_pedidos.csv / detalle_pedidos | Una línea de pedido | id_detalle | 10.681 |

Relaciones: `clientes → pedidos` por `id_cliente`; `pedidos → detalle_pedidos` por `id_pedido`; `productos → detalle_pedidos` por `id_producto`. Son relaciones de uno a muchos, cuya integridad comprobaremos.

**GitHub:** guarda este notebook junto a la carpeta `data/` del ZIP. GitHub permite leer el notebook; para ejecutarlo necesitas Jupyter/VS Code con PySpark y Java compatibles, o Microsoft Fabric. En local, instala PySpark en tu entorno (`python -m pip install pyspark==3.5.3`) y selecciona ese kernel. No necesitas pandas para esta guía.

**Fabric:** importa el `.ipynb`, usa el lenguaje PySpark, adjunta un Lakehouse y sube los cuatro CSV a `Files/spark_sql/data/`. Cambia `BASE_DATOS` a esa ruta en la siguiente celda. Usa una sesión Spark con acceso al Lakehouse.

El notebook usa `spark.sql(...)` para ejecutar SQL desde celdas Python: así conserva un único kernel y funciona en Jupyter y Fabric. En Fabric también puedes copiar el contenido SQL a una celda con `%%sql` en su primera línea. No copies la envoltura Python a una celda SQL.

La parte de lectura crea **vistas temporales**: no modifica tus CSV ni crea tablas persistentes. Al cerrar la sesión, las vistas desaparecen; vuelve a ejecutar la carga.


In [ ]:
from pyspark.sql import SparkSession

# En Fabric la sesión spark suele existir; en local se crea aquí.
spark = SparkSession.builder.appName("Spark_SQL_Referencia").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

BASE_DATOS = "data"  # Local: carpeta junto al notebook.
# BASE_DATOS = "Files/spark_sql/data"  # Fabric, Lakehouse predeterminado.
# También puedes usar una ruta absoluta o una ruta ABFS copiada de Fabric.

ACTIVAR_DELTA = False  # Solo True en un entorno compatible con Delta.
ACTIVAR_CLON = False   # Solo True si el runtime admite SHALLOW CLONE.

# Devuelve el DataFrame para reutilizarlo y muestra el resultado.
def sql(consulta, filas=20):
    resultado = spark.sql(consulta)
    resultado.show(filas, truncate=False)
    return resultado

print("Spark:", spark.version)


### Cargar con tipos explícitos
El código postal y los identificadores son texto: así se conservan ceros iniciales. Los importes usan `DECIMAL`, las fechas `DATE`, las cantidades `INT` y `activo` es `BOOLEAN`. `descuento_pct = 0.20` significa un 20 %, no un 0,20 %.

`FAILFAST` detiene la carga ante registros mal formados. Si cambia la estructura de un CSV, revisa primero su cabecera y su esquema.

In [ ]:
ESQUEMAS = {
    "clientes": "id_cliente STRING, nombre STRING, apellidos STRING, email STRING, telefono STRING, ciudad STRING, provincia STRING, codigo_postal STRING, fecha_alta DATE, segmento STRING",
    "productos": "id_producto STRING, nombre_producto STRING, categoria STRING, precio_unitario DECIMAL(12,2), coste_unitario DECIMAL(12,2), activo BOOLEAN",
    "pedidos": "id_pedido STRING, id_cliente STRING, fecha_pedido DATE, canal STRING, estado STRING, importe_total DECIMAL(14,2)",
    "detalle_pedidos": "id_detalle STRING, id_pedido STRING, numero_linea INT, id_producto STRING, cantidad INT, precio_unitario DECIMAL(12,2), descuento_pct DECIMAL(5,2), importe_linea DECIMAL(14,2)"
}
for nombre, esquema in ESQUEMAS.items():
    ruta = f"{BASE_DATOS.rstrip('/')}/{nombre}.csv"
    df = (spark.read.option("header", True).option("encoding", "UTF-8")
          .option("mode", "FAILFAST").option("dateFormat", "yyyy-MM-dd")
          .schema(esquema).csv(ruta))
    df.createOrReplaceTempView(nombre)
    print(nombre, df.count(), "filas")


## 2. Explorar y seleccionar
Las consultas usan las vistas sin prefijo `dbo`. Para tablas persistentes de tu Lakehouse, utiliza el nombre de esquema y tabla correspondiente a tu entorno.

### SHOW TABLES y DESCRIBE

Lista los objetos visibles; `isTemporary` identifica las vistas de esta sesión.

In [ ]:
sql("""
SHOW TABLES
""")

In [ ]:
sql("DESCRIBE TABLE pedidos")

### SELECT, alias, ORDER BY y LIMIT

Los 10 pedidos de mayor importe. El segundo criterio resuelve empates. En Spark SQL usa `LIMIT`, no `TOP`.

In [ ]:
sql("""
SELECT id_pedido, id_cliente, fecha_pedido, canal, importe_total AS importe
FROM pedidos
ORDER BY importe DESC, id_pedido
LIMIT 10
""")

### DISTINCT

Valores distintos; no elimina filas de la fuente.

In [ ]:
sql("""
SELECT DISTINCT canal, estado FROM pedidos ORDER BY canal, estado
""")

## 3. Filtrar, transformar y tratar nulos

### WHERE, AND, OR, IN y BETWEEN

`BETWEEN` incluye ambos extremos. Los paréntesis evitan ambigüedad entre AND y OR.

In [ ]:
sql("""
SELECT id_pedido, canal, estado, importe_total
FROM pedidos
WHERE canal IN ('web', 'app')
  AND importe_total BETWEEN 100 AND 500
  AND (estado = 'entregado' OR estado = 'enviado')
ORDER BY importe_total DESC, id_pedido LIMIT 20
""")

### LIKE y cadenas

`%` representa cualquier secuencia. `LOWER` facilita búsquedas sin distinguir mayúsculas.

In [ ]:
sql("""
SELECT id_producto, UPPER(nombre_producto) AS producto, TRIM(categoria) AS categoria
FROM productos WHERE LOWER(nombre_producto) LIKE '%premium%'
ORDER BY id_producto
""")

### NULL, COALESCE y CAST

Un nulo se comprueba con IS NULL, nunca con = NULL. COALESCE devuelve el primer valor no nulo.

In [ ]:
sql("""
SELECT id_cliente, COALESCE(ciudad, 'Sin ciudad') AS ciudad,
       CAST(codigo_postal AS STRING) AS codigo_postal
FROM clientes WHERE ciudad IS NOT NULL ORDER BY id_cliente LIMIT 10
""")

### CASE WHEN

Clasifica pedidos mediante reglas ordenadas.

In [ ]:
sql("""
SELECT id_pedido, importe_total,
 CASE WHEN importe_total >= 1000 THEN 'Alto'
      WHEN importe_total >= 200 THEN 'Medio' ELSE 'Bajo' END AS tramo
FROM pedidos ORDER BY importe_total DESC, id_pedido LIMIT 15
""")

### Fechas

Ejemplo con fechas fijas reproducibles. DATEDIFF devuelve días entre dos fechas.

In [ ]:
sql("""
SELECT id_pedido, fecha_pedido, YEAR(fecha_pedido) AS anio,
 MONTH(fecha_pedido) AS mes, DATE_FORMAT(fecha_pedido, 'yyyy-MM') AS periodo,
 DATEDIFF(DATE '2026-11-13', fecha_pedido) AS dias_hasta_referencia
FROM pedidos
WHERE fecha_pedido >= DATE '2026-01-01' AND fecha_pedido < DATE '2027-01-01'
ORDER BY fecha_pedido, id_pedido LIMIT 10
""")

## 4. Agregar y unir tablas
En esta guía, los importes por estado se muestran tal como están en los CSV. Para hablar de ventas entregadas filtramos `estado = 'entregado'`; sumar todos los estados incluye cancelados y pendientes.

### COUNT y UNION ALL

UNION ALL apila resultados sin eliminar duplicados; UNION sí los elimina.

In [ ]:
sql("""
SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM clientes
UNION ALL
SELECT 'productos' AS tabla, COUNT(*) AS filas FROM productos
UNION ALL
SELECT 'pedidos' AS tabla, COUNT(*) AS filas FROM pedidos
UNION ALL
SELECT 'detalle_pedidos' AS tabla, COUNT(*) AS filas FROM detalle_pedidos
""")

### GROUP BY y agregados

COUNT(*) cuenta filas; COUNT(columna) solo cuenta valores no nulos.

In [ ]:
sql("""
SELECT canal, COUNT(*) AS num_pedidos,
 SUM(importe_total) AS importe, ROUND(AVG(importe_total), 2) AS ticket_medio,
 MIN(importe_total) AS minimo, MAX(importe_total) AS maximo
FROM pedidos WHERE estado = 'entregado'
GROUP BY canal ORDER BY importe DESC
""")

### HAVING

WHERE filtra filas antes de agrupar; HAVING filtra los grupos ya calculados.

In [ ]:
sql("""
SELECT id_cliente, COUNT(*) AS num_pedidos, SUM(importe_total) AS importe
FROM pedidos WHERE estado = 'entregado'
GROUP BY id_cliente HAVING COUNT(*) >= 10
ORDER BY importe DESC, id_cliente LIMIT 10
""")

### INNER JOIN: ventas por provincia

Solo conserva pedidos con un cliente coincidente. Se une por identificador, nunca por nombre.

In [ ]:
sql("""
SELECT c.provincia, COUNT(*) AS num_pedidos, SUM(p.importe_total) AS importe
FROM pedidos p INNER JOIN clientes c ON p.id_cliente = c.id_cliente
WHERE p.estado = 'entregado' GROUP BY c.provincia ORDER BY importe DESC
""")

### LEFT JOIN: incluir clientes sin pedidos

Conserva todos los clientes. COUNT(p.id_pedido) evita contar la fila vacía del cliente sin pedidos.

In [ ]:
sql("""
SELECT c.id_cliente, c.nombre, COUNT(p.id_pedido) AS num_pedidos,
 COALESCE(SUM(p.importe_total), 0) AS importe
FROM clientes c LEFT JOIN pedidos p
 ON c.id_cliente = p.id_cliente AND p.estado = 'entregado'
GROUP BY c.id_cliente, c.nombre ORDER BY num_pedidos, c.id_cliente LIMIT 15
""")

### Productos con más unidades

Agrupa también por id_producto: dos productos pueden compartir nombre. Usa importe_linea para el importe de detalle; no sumes importe_total tras unir cabeceras y líneas, porque se repetiría por cada línea.

In [ ]:
sql("""
SELECT pr.id_producto, pr.nombre_producto, pr.categoria,
 SUM(d.cantidad) AS unidades, SUM(d.importe_linea) AS importe
FROM detalle_pedidos d
JOIN productos pr ON d.id_producto = pr.id_producto
JOIN pedidos p ON d.id_pedido = p.id_pedido
WHERE p.estado = 'entregado'
GROUP BY pr.id_producto, pr.nombre_producto, pr.categoria
ORDER BY unidades DESC, pr.id_producto LIMIT 5
""")

## 5. CTE y funciones de ventana

### WITH: una CTE para ordenar el razonamiento

Una CTE es un resultado con nombre dentro de una consulta; no crea una tabla persistente.

In [ ]:
sql("""
WITH resumen AS (
 SELECT id_cliente, SUM(importe_total) AS importe
 FROM pedidos WHERE estado = 'entregado' GROUP BY id_cliente
)
SELECT c.id_cliente, c.nombre, r.importe
FROM resumen r JOIN clientes c ON r.id_cliente = c.id_cliente
ORDER BY r.importe DESC, c.id_cliente LIMIT 10
""")

### ROW_NUMBER: último pedido por cliente

PARTITION BY separa clientes; ORDER BY determina el orden dentro de cada cliente.

In [ ]:
sql("""
WITH ordenados AS (
 SELECT *, ROW_NUMBER() OVER (
 PARTITION BY id_cliente ORDER BY fecha_pedido DESC, id_pedido DESC) AS posicion
 FROM pedidos
)
SELECT id_cliente, id_pedido, fecha_pedido, importe_total
FROM ordenados WHERE posicion = 1 ORDER BY id_cliente LIMIT 15
""")

### SUM OVER y LAG

Primero se agrega por mes; después se calcula acumulado y mes anterior. Si falta un mes, LAG devuelve el anterior disponible, no necesariamente el mes natural anterior.

In [ ]:
sql("""
WITH mensual AS (
 SELECT DATE_FORMAT(fecha_pedido, 'yyyy-MM') AS mes, SUM(importe_total) AS importe
 FROM pedidos WHERE estado = 'entregado'
 GROUP BY DATE_FORMAT(fecha_pedido, 'yyyy-MM')
)
SELECT mes, importe,
 LAG(importe) OVER (ORDER BY mes) AS importe_mes_anterior_disponible,
 SUM(importe) OVER (ORDER BY mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS acumulado
FROM mensual ORDER BY mes
""")

## 6. Calidad de datos
Una consulta de incidencias sin filas es un buen resultado. No se corrigen ni borran registros automáticamente.

### Identificadores duplicados

El resultado debe estar vacío si id_pedido es único. Repite el patrón para las otras claves.

In [ ]:
sql("""
SELECT id_pedido, COUNT(*) AS repeticiones FROM pedidos
GROUP BY id_pedido HAVING COUNT(*) > 1
""")

### Campos obligatorios y valores inválidos

Cuenta nulos y negativos sin modificar la fuente.

In [ ]:
sql("""
SELECT COUNT(*) AS filas,
 SUM(CASE WHEN id_pedido IS NULL THEN 1 ELSE 0 END) AS sin_id,
 SUM(CASE WHEN fecha_pedido IS NULL THEN 1 ELSE 0 END) AS sin_fecha,
 SUM(CASE WHEN importe_total IS NULL OR importe_total < 0 THEN 1 ELSE 0 END) AS importes_invalidos
FROM pedidos
""")

### LEFT ANTI JOIN: referencias inexistentes

Devuelve pedidos cuyo cliente no existe.

In [ ]:
sql("""
SELECT p.id_pedido, p.id_cliente FROM pedidos p
LEFT ANTI JOIN clientes c ON p.id_cliente = c.id_cliente
""")

### Conciliar cabeceras y líneas

Se comparan importes redondeados a céntimos, incluyendo pedidos sin líneas.

In [ ]:
sql("""
WITH lineas AS (
 SELECT id_pedido, SUM(importe_linea) AS total_lineas FROM detalle_pedidos GROUP BY id_pedido
)
SELECT p.id_pedido, p.importe_total, l.total_lineas,
 p.importe_total - l.total_lineas AS diferencia
FROM pedidos p LEFT JOIN lineas l ON p.id_pedido = l.id_pedido
WHERE l.id_pedido IS NULL OR ROUND(p.importe_total - l.total_lineas, 2) <> 0
ORDER BY p.id_pedido
""")

### Recalcular descuentos

Valida la fórmula cantidad × precio × (1 − descuento).

In [ ]:
sql("""
SELECT id_detalle, importe_linea,
 ROUND(cantidad * precio_unitario * (1 - descuento_pct), 2) AS calculado
FROM detalle_pedidos
WHERE importe_linea <> ROUND(cantidad * precio_unitario * (1 - descuento_pct), 2)
ORDER BY id_detalle LIMIT 20
""")

## 7. Delta: escritura, historial y clonación (opcional)
**Solo para Fabric o Spark configurado con Delta Lake.** Instalar PySpark por sí solo no añade Delta. Estas celdas se omiten mientras `ACTIVAR_DELTA = False`.

Se conserva la práctica de tu original, pero se usa una tabla nueva con nombre único por ejecución. Nunca se escribirá sobre `clientes`, `productos`, `pedidos` o `detalle_pedidos`. La tabla de práctica queda guardada hasta que la elimines expresamente.

`USING DELTA` selecciona el formato de tabla. `INSERT` añade filas; `UPDATE` modifica filas; `DELETE` elimina filas. Reejecuta el bloque completo desde su primera celda para crear una nueva práctica reproducible.


In [ ]:
import uuid
if ACTIVAR_DELTA:
    # Para un Lakehouse con esquemas puedes cambiarlo a "dbo.".
    ESQUEMA_PRACTICA = ""
    tabla_practica = ESQUEMA_PRACTICA + "practica_sql_" + uuid.uuid4().hex[:10]
    sql(f"CREATE TABLE {tabla_practica} (id INT, importe_centimos INT) USING DELTA")
    sql(f"INSERT INTO {tabla_practica} VALUES (1, 1000), (2, 2000)")
    version_inicial = spark.sql(f"DESCRIBE HISTORY {tabla_practica}").orderBy("version", ascending=False).first()["version"]
    sql(f"SELECT * FROM {tabla_practica} ORDER BY id")
    print("Tabla:", tabla_practica, "Versión inicial con datos:", version_inicial)
else:
    print("Delta omitido. Actívalo solo en un entorno compatible.")


### Metadatos, UPDATE y DELETE
`DESCRIBE DETAIL` muestra ubicación, formato, tamaño y `numFiles`. `DESCRIBE HISTORY` muestra las operaciones y versiones; no representa una copia completa nueva de todos los archivos en cada cambio.

In [ ]:
if ACTIVAR_DELTA:
    sql(f"DESCRIBE DETAIL {tabla_practica}")
    sql(f"UPDATE {tabla_practica} SET importe_centimos = 1500 WHERE id = 1")
    sql(f"DELETE FROM {tabla_practica} WHERE id = 2")
    sql(f"SELECT * FROM {tabla_practica} ORDER BY id")
    sql(f"DESCRIBE HISTORY {tabla_practica} LIMIT 10")


### Time travel: comparar estados
La versión se obtiene del historial real, evitando marcadores no ejecutables como `<version_inicial>`. Deben estar disponibles el registro y los archivos de esa versión. La retención y VACUUM pueden impedir consultar versiones antiguas. El tiempo se configura en UTC.

In [ ]:
if ACTIVAR_DELTA:
    sql(f"SELECT * FROM {tabla_practica} VERSION AS OF {version_inicial} ORDER BY id")
    sql(f"SELECT * FROM {tabla_practica} ORDER BY id")
    # Alternativa por timestamp: usar el instante exacto de la versión inicial.
    instante = (spark.sql(f"DESCRIBE HISTORY {tabla_practica}")
                .where(f"version = {version_inicial}")
                .selectExpr("date_format(timestamp, 'yyyy-MM-dd HH:mm:ss.SSS') AS instante")
                .first()["instante"])
    sql(f"SELECT * FROM {tabla_practica} TIMESTAMP AS OF '{instante}' ORDER BY id")


### SHALLOW CLONE
Requiere un runtime que admita esta instrucción. Un clon superficial tiene historial propio, pero referencia archivos del origen: **no es una copia de seguridad independiente**. No se sincroniza automáticamente con los cambios posteriores del origen. Conserva los archivos de origen mientras el clon los necesite.

In [ ]:
if ACTIVAR_DELTA and ACTIVAR_CLON:
    tabla_clon = tabla_practica + "_clon"
    sql(f"CREATE TABLE {tabla_clon} SHALLOW CLONE {tabla_practica} VERSION AS OF {version_inicial}")
    sql(f"UPDATE {tabla_clon} SET importe_centimos = 9000 WHERE id = 1")
    sql(f"SELECT * FROM {tabla_clon} ORDER BY id")
    sql(f"SELECT * FROM {tabla_practica} ORDER BY id")
    sql(f"DESCRIBE HISTORY {tabla_clon} LIMIT 10")


### Limpieza voluntaria
Solo elimina las tablas de esta práctica. Primero el clon y después el origen. Se deja desactivada para poder revisar los resultados.

In [ ]:
ELIMINAR_PRACTICA = False
if ACTIVAR_DELTA and ELIMINAR_PRACTICA:
    if ACTIVAR_CLON:
        sql(f"DROP TABLE IF EXISTS {tabla_practica}_clon")
    sql(f"DROP TABLE IF EXISTS {tabla_practica}")


## 8. Spark SQL frente al endpoint SQL del Lakehouse

| Tema | Este notebook (Spark SQL) | Endpoint SQL del Lakehouse (T-SQL) |
|---|---|---|
| Limitar filas | `SELECT ... LIMIT 10` | `SELECT TOP 10 ...` |
| Fuente de esta guía | Vistas temporales sobre CSV | Tablas Delta persistentes sincronizadas |
| Vistas temporales de Spark | Visibles en esta sesión | No se publican en el endpoint |
| INSERT / UPDATE / DELETE sobre Delta | En Spark con soporte Delta | No admite estas escrituras sobre tablas del Lakehouse |
| Historial y versiones Delta | `DESCRIBE HISTORY`, `VERSION AS OF` | No copiar estas instrucciones Spark al endpoint |

Para repetir tu ejercicio del endpoint, crea primero la tabla Delta opcional en Spark y consulta **su nombre real** desde el endpoint una vez sincronizada. Un intento de INSERT allí se espera que sea rechazado; no se incluye como celda ejecutable porque este notebook corre en Spark y allí sí podría escribir. Esta limitación se refiere al endpoint del **Lakehouse**, no a Fabric Warehouse.

### Errores frecuentes

| Error o síntoma | Qué revisar |
|---|---|
| No se encuentra el CSV | `BASE_DATOS`, carpeta de trabajo y Lakehouse adjunto |
| No se encuentra pedidos | Ejecutar primero la carga en la misma sesión |
| Error cerca de TOP | Sustituir por LIMIT al final de la consulta Spark |
| Error con dbo | Verificar el esquema real; las vistas de esta guía no llevan dbo |
| Delta no disponible | Mantener el bloque opcional desactivado o usar un runtime compatible |
| Ventas demasiado altas tras un JOIN | No sumar importe_total cuando el pedido se repite en cada línea |
| Fechas o números mal interpretados | Revisar esquema, formato de fecha y separador decimal |

## 9. Ejercicios para practicar
1. Obtén los 10 productos activos de mayor precio, mostrando su categoría.
2. Calcula el importe de pedidos entregados por año y canal.
3. Encuentra productos sin ninguna línea de pedido con LEFT ANTI JOIN.
4. Obtén el pedido de mayor importe de cada cliente usando ROW_NUMBER.
5. En Delta, explica por qué la versión inicial tiene dos filas y la actual una.

**Cómo comprobarte:** ordena los resultados de forma determinista; contrasta los totales con una consulta agregada independiente; revisa que no se dupliquen las claves al unir tablas. En el ejercicio Delta, antes de limpiar, la versión inicial suma 3.000 céntimos y la actual 1.500.

### Chuleta

| Necesito… | Comando o patrón |
|---|---|
| Explorar objetos | SHOW TABLES / DESCRIBE TABLE |
| Elegir columnas | SELECT columnas FROM tabla |
| Filtrar filas | WHERE / IN / BETWEEN / LIKE / IS NULL |
| Ordenar y limitar | ORDER BY ... LIMIT n |
| Eliminar duplicados del resultado | DISTINCT |
| Resumir | COUNT / SUM / AVG / MIN / MAX + GROUP BY |
| Filtrar grupos | HAVING |
| Unir datos por clave | INNER JOIN / LEFT JOIN |
| Encontrar sin coincidencia | LEFT ANTI JOIN |
| Apilar resultados | UNION ALL |
| Reemplazar nulos / clasificar | COALESCE / CASE WHEN |
| Organizar consultas | WITH (CTE) |
| Calcular por partición | ROW_NUMBER / LAG / SUM OVER |
| Escribir en Delta | CREATE TABLE USING DELTA / INSERT / UPDATE / DELETE |
| Consultar versiones | DESCRIBE HISTORY / VERSION AS OF / TIMESTAMP AS OF |

### Fuentes oficiales y mantenimiento
Documentación consultada el 13/11/2026. La compatibilidad depende del runtime; revisa las fuentes antes de trasladar el bloque avanzado a otro motor.
- [Apache Spark: comenzar con Spark SQL](https://spark.apache.org/docs/3.5.3/sql-getting-started.html)
- [Apache Spark: sintaxis SQL](https://spark.apache.org/docs/3.5.3/sql-ref-syntax.html)
- [Microsoft Fabric: endpoint SQL del Lakehouse](https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-sql-analytics-endpoint)
- [Microsoft Fabric: clonar tablas Delta](https://learn.microsoft.com/en-us/fabric/data-engineering/delta-lake-clone)
- [Delta Lake: historial, time travel y utilidades](https://docs.delta.io/delta-utility/)

**Para GitHub:** este cuaderno se entrega sin salidas guardadas, identificadores de workspace ni credenciales. Antes de publicar los CSV, confirma que son datos de práctica y que puedes compartirlos; la tabla clientes contiene nombres, correos y teléfonos. Puedes usar un repositorio privado si necesitas conservarlos sin publicarlos.
